## Setup

In [1]:
from pathlib import Path
import re

import pandas as pd
from tqdm.auto import tqdm


tqdm.pandas()  # load tqdm's pandas support
pd.set_option("display.max_colwidth", None)

In [2]:
from sglang.srt.server import launch_server  # for testing

In [3]:
repo_id = "meta-llama/Llama-3.2-1B-Instruct"

# from datasets import load_dataset
# from huggingface_hub import InferenceClient
# llm_client = InferenceClient(model=repo_id, timeout=120)
# llm_client.text_generation(prompt="How are you today? ", max_new_tokens=20)

print(
    f"Run:\n\npython -m sglang.launch_server --model-path {repo_id} --port 30000 --host 0.0.0.0"
)

Run:

python -m sglang.launch_server --model-path meta-llama/Llama-3.2-1B-Instruct --port 30000 --host 0.0.0.0


## Prepare the data

In [8]:
base = Path(".").resolve() / "eval-all" / "data"
assert base.is_dir() and base.exists()
base


PosixPath('/workspaces/ts-qa/eval-all/data')

In [9]:
df = pd.read_hdf(base / "llmjudge_eval_set.h5", key="data")
df

,setting,model,mode,sample_id,question_id,prediction_text,action_sequence,textual_description,question_type,question,answer_type,answer_text
271884,4.2.1,Falcon 2 (11B),450,27376,4,"running, throwing a ball, dancing","{'start': [0.0, 4.0, 8.0, 12.0], 'end': [4.0, 8.0, 12.0, 16.0], 'action': ['running', 'throwing a ball', 'running', 'dancing'], 'action_sentence': ['The activity running is the first action done by the person', 'The person is engaging in throwing a ball', 'Someone is running', 'As the last action, the person is performing dancing']}","The activity running is the first action done by the person from 0:0.0 to 0:4.0 (4.0 seconds total). The person is engaging in throwing a ball. This is happening from 0:4.0 to 0:8.0, which means for a total time of 4.0 seconds. Someone is running from 0:8.0 to 0:12.0 (for 4.0 seconds). As the last action, the person is performing dancing. This action is starting at 0:12.0 and happens until 0:16.0, meaning the engagement in this is for 4.0 seconds.",count_number_open,Which action does the person carry out 2 times?,open,running
1259842,4.2.1,Llama 3 (8B),400,26968,2,1,"{'start': [0.0, 4.0, 8.0, 12.0], 'end': [4.0, 8.0, 12.0, 16.0], 'action': ['throwing a ball', 'jumping once', 'punching', 'punching'], 'action_sentence': ['The sequence starts with throwing a ball', 'The person is jumping once', 'The person is engaging in punching', 'The last action is punching']}","The sequence starts with throwing a ball at timestamp 0:0.0 and ending at 0:4.0, resulting in a total time of 4.0 seconds. The person is jumping once. This action is starting at 0:4.0 and continues until 0:8.0, resulting in a total time of 4.0 seconds. The person is engaging in punching. This is happening from 0:8.0 to 0:12.0, which means for a total time of 4.0 seconds. The last action is punching at timestamp 0:12.0 and ending at 0:16.0, resulting in a total time of 4.0 seconds.",count_open,How frequently does the individual partake in punching?,open,1
1984164,4.2.1,T5 Small,100,24832,4,shaking hands,"{'start': [0.0, 4.0, 8.0, 12.0], 'end': [4.0, 8.0, 12.0, 16.0], 'action': ['kicking a ball', 'shaking hands', 'T-posing', 'T-posing'], 'action_sentence': ['As the first action, the person is kicking a ball', 'The activity is shaking hands', 'The activity is T-posing', 'The finishing action of the person is T-posing']}","As the first action, the person is kicking a ball. This is happening from 0:0.0 to 0:4.0, which means for a total time of 4.0 seconds. The activity is shaking hands from 0:4.0 to 0:8.0 (for 4.0 seconds). The activity is T-posing. This action is starting at 0:8.0 and continues until 0:12.0, resulting in a total time of 4.0 seconds. The finishing action of the person is T-posing at timestamp 0:12.0 and ending at 0:16.0, resulting in a total time of 4.0 seconds.",last_open,What was the action performed as the final action by the person?,open,Someone is T-posing.
1838024,4.2.1,T5 Base,150,28604,4,waving,"{'start': [0.0, 4.0, 8.0, 12.0], 'end': [4.0, 8.0, 12.0, 16.0], 'action': ['jumping once', 'jumping once', 'bowing', 'waving'], 'action_sentence': ['The activity jumping once is the first action done by the person', 'The activity is jumping once', 'The person is bowing', 'The person is terminating the sequence with waving']}","The activity jumping once is the first action done by the person. This action is starting at 0:0.0 and happens until 0:4.0, meaning the engagement in this is for 4.0 seconds. The activity is jumping once. It is starting at second 0:4.0 and continuing until 0:8.0, so the person is doing this for 4.0 seconds. The person is bowing. This action is starting at 0:8.0 and continues until 0:12.0, resulting in a total time of 4.0 seconds. The person is terminating the sequence with waving. This action is starting at 0:12.0 and continues until 0:16.0, resulting in a total time of 4.0 seconds.",right_after_open,What did the person do right after engaging in bowing?,open,Someone is waving.
1583837,4.2.1

In [16]:
JUDGE_PROMPT = """You will be given a scene_description with timesamps and a scene_question regarding that scene. You will also be given a reference_answer and system_answer couple.
Your task is to provide a TotalRating scoring how well the system_answer answers the user concerns expressed in the scene_question. The reference_answer is provided as reference for a very good answer.
Give your answer on a scale of 1 to 4, where 1 means that the system_answer is not helpful at all, and 4 means that the system_answer completely and helpfully addresses the scene_question.
You will also provide a brief rationale for your rating.

Here is the scale you should use to build your answer:
1: The system_answer is terrible: completely irrelevant to the question asked, or corrupt in some way (e.g., empty or nonsensical)
2: The system_answer is bad: misses some key aspects of the question or is factually incorrect
3: The system_answer is good: correctly answers the scene_question and is helpful in the context of the scene

Provide your feedback in the following JSON format:
{{
    "BriefRationale:": "<your rationale, at most 3 sentences>",
    "TotalRating:": <your rating>
}}

You MUST provide values for "BriefRationale" and "TotalRating" in your answer.
Do NOT respond with any other text in your answer.

Now here are the question and answer:

- scene_description: {scene_description}
- scene_question: {scene_question}
- reference_answer: {reference_answer}
- system_answer: {system_answer}

Provide your feedback. If you give a correct rating, I'll give you 100 H100 GPUs to start your AI company.

"""


def make_judge_prompt(row) -> str:
    # scene_description = TODO make table from this
    scene_description = row["textual_description"]

    return JUDGE_PROMPT.format(
        scene_description=scene_description,
        scene_question=row["question"],
        reference_answer=row["answer_text"],
        system_answer=row["prediction_text"],
    )


df["llm_judge"] = df.progress_apply(
    lambda x: llm_client.text_generation(
        prompt=make_judge_prompt(x),
        max_new_tokens=500,  # TODO temperature=0.5
    ),
    axis=1,
)


def extract_judge_score(answer: str, split_str: str = "Total rating:") -> int | None:
    try:
        if split_str in answer:
            rating = answer.split(split_str)[1]
        else:
            rating = answer
        digit_groups = [el.strip() for el in re.findall(r"\d+(?:\.\d+)?", rating)]
        return int(digit_groups[0])
    except Exception as e:
        print(e)
        return pd.NA


df["llm_judge_score"] = df["llm_judge_improved"].apply(extract_judge_score)

# Adjust the score to be in the same range as the human scores
df["llm_judge_score"] *= 3

print("Correlation between LLM-as-a-judge and the human raters:")
print(f"{df['llm_judge_improved_score'].corr(df['human_score'], method='pearson'):.3f}")

  0%|          | 0/200 [00:00<?, ?it/s]

KeyError: 'llm_judge_improved'